In [1]:
import requests

#Get Teams and players data

# API, Token e request
url = "http://api.football-data.org/v4/competitions/2000/teams"
headers = {"X-Auth-Token": "906be5946ca94b089a829d205e689da8"}
response = requests.get(url, headers=headers)

data = response.json()
teams = data.get("teams", [])

StatementMeta(, 4ab20ea1-1e21-4b12-b087-7c98916b7af6, 3, Finished, Available, Finished, False)

In [2]:
import pandas as pd

pandasdf = pd.DataFrame(teams, columns=["id", "name", "shortName", "tla", "crest", "squad"])

sparkdf = spark.createDataFrame(pandasdf)
sparkdf.write.mode('overwrite').saveAsTable('bronze_Teams')

StatementMeta(, 4ab20ea1-1e21-4b12-b087-7c98916b7af6, 4, Finished, Available, Finished, False)

In [3]:
import requests

#Get Matches data

# API, Token e request
url_matches = "http://api.football-data.org/v4/competitions/2000/matches?season=2026"
headers = {"X-Auth-Token": "906be5946ca94b089a829d205e689da8"}
response_matches = requests.get(url_matches, headers=headers)

data_matches = response_matches.json()
matches = data_matches.get("matches", [])

StatementMeta(, 4ab20ea1-1e21-4b12-b087-7c98916b7af6, 5, Finished, Available, Finished, False)

In [4]:
import pandas as pd

# Converter para DataFrame
pandasdf_matches = pd.DataFrame(matches, columns=["id", "utcDate", "matchDay", "stage", "homeTeam", "awayTeam", "score"])

# display(pandasdf_matches)

sparkdf_matches = spark.createDataFrame(pandasdf_matches)
sparkdf_matches.write.mode('overwrite').saveAsTable('bronze_Matches')

StatementMeta(, 4ab20ea1-1e21-4b12-b087-7c98916b7af6, 6, Finished, Available, Finished, False)

In [5]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Seu código original
pandasdf_matches = pd.DataFrame(matches, columns=["id", "utcDate", "matchDay", "stage", "homeTeam", "awayTeam", "score"])

# Definimos o esquema exato que o Spark deve usar para ler o Pandas
schema_matches = StructType([
    StructField("id", LongType(), True),
    StructField("utcDate", StringType(), True),
    StructField("matchDay", DoubleType(), True),
    StructField("stage", StringType(), True),
    # O Spark mapeia dicionários do Pandas para structs automaticamente se passarmos a estrutura:
    StructField("homeTeam", StructType([
        StructField("crest", StringType(), True),
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("shortName", StringType(), True),
        StructField("tla", StringType(), True)
    ]), True),
    StructField("awayTeam", StructType([
        StructField("crest", StringType(), True),
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("shortName", StringType(), True),
        StructField("tla", StringType(), True)
    ]), True),
    # Aqui forçamos os campos problemáticos a serem salvos como String ou Long, eliminando o 'void'
    StructField("score", StructType([
        StructField("duration", StringType(), True),
        StructField("fullTime", StructType([
            StructField("away", StringType(), True), # Forçado para String
            StructField("home", StringType(), True)  # Forçado para String
        ]), True),
        StructField("halfTime", StructType([
            StructField("away", StringType(), True), # Forçado para String
            StructField("home", StringType(), True)  # Forçado para String
        ]), True),
        StructField("winner", StringType(), True)    # Forçado para String
    ]), True)
])

# Criamos o dataframe injetando o Schema
sparkdf_matches = spark.createDataFrame(pandasdf_matches, schema=schema_matches)

# Usamos o overwriteSchema na escrita para limpar o histórico ruim da tabela Delta anterior
sparkdf_matches.write \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable('bronze_Matches')

StatementMeta(, 4ab20ea1-1e21-4b12-b087-7c98916b7af6, 7, Finished, Available, Finished, False)